In [47]:
if(!require(pacman)) install.packages(pacman)

### **Parte 1A - Módulo de Tokenização + Remoção de Stopwords:**

In [48]:
pacman::p_load(stopwords)

# Os códigos abaixo giram em torno
# da tokenização de strings e a remoção
# de stopwords.

remove_punctuation <- function(txt) {
  txt <- tolower(txt)
  txt <- gsub("[[:punct:]]", "", txt)
  return(txt)
}

tokenize_txt <- function(txt) {
  words_from_txt <- unlist(strsplit(txt, "\\s+"))
  words_from_txt <- words_from_txt[words_from_txt != ""]
  return(words_from_txt)
}

remove_stopwords <- function(txt, desligado = FALSE) {
  token_from_txt <- tokenize_txt(remove_punctuation(txt))
  
  stopwords_set_1 <- stopwords::stopwords(language = "en", source = "snowball")
  stopwords_set_2 <- stopwords::stopwords(language = "pt", source = "snowball")
  stopwords_multilingual <- unique(c(stopwords_set_1, stopwords_set_2))
  
  if (desligado == FALSE) {
    token_without_stopwords <- token_from_txt[!token_from_txt %in% stopwords_multilingual]
    return(token_without_stopwords)
  } else {
    return(token_from_txt)
  }
}

### **Parte 1B - Testando Módulo de Tokenização + Remoção de Stopwords:**

In [49]:
sample_txts = c(
    'O gato pulou na mesa.',
    'The quick brown fox jumps over the lazy dog.'   
)

In [50]:
tokenize_txt(sample_txts[1])
tokenize_txt(sample_txts[2])

[1] "O"     "gato"  "pulou" "na"    "mesa."

[1] "The"   "quick" "brown" "fox"   "jumps" "over"  "the"   "lazy"  "dog."

In [51]:
remove_stopwords(txt = sample_txts[1],desligado = TRUE)
remove_stopwords(txt = sample_txts[1],desligado = FALSE)

[1] "o"     "gato"  "pulou" "na"    "mesa"

[1] "gato"  "pulou" "mesa"

In [52]:
remove_stopwords(txt = sample_txts[2],desligado = TRUE)
remove_stopwords(txt = sample_txts[2],desligado = FALSE)

[1] "the"   "quick" "brown" "fox"   "jumps" "over"  "the"   "lazy"  "dog"

[1] "quick" "brown" "fox"   "jumps" "lazy"  "dog"

### **Parte 2A - Módulo de criação das marizes Bag of Words (BoW) e TF-IDF:**

In [53]:
# Os códigos abaixo giram em torno
# de manusear textos já preprocessados,
# e fazer coisas como: (1) matriz BoW, (2) matriz TF-IDF.
#
# Diferente da primeira versão, aqui trabalhamos DIRETO com os tokens
# (sem reconstruir string e tokenizar de novo), já que não dependemos
# de nenhum vectorizer externo.

get_processed_tokens <- function(nonProcessedTexts,
                                  dontRemoveStopWords = FALSE) {
  lapply(nonProcessedTexts, function(txt) {
    remove_stopwords(txt, desligado = dontRemoveStopWords)
  })
}

get_BOW_matrix <- function(nonProcessedTexts,
                            dontRemoveStopWords = FALSE) {

  tokenized_corpus <- get_processed_tokens(nonProcessedTexts, dontRemoveStopWords)

  # vocabulário: todas as palavras únicas em todos os documentos
  vocabulary <- sort(unique(unlist(tokenized_corpus)))

  # nomes das colunas: recriamos as frases só para rotular a matriz
  # (não são usadas em nenhum processamento, é só exibição)
  doc_labels <- sapply(tokenized_corpus, paste, collapse = " ")

  bow_matrix <- matrix(0,
                        nrow = length(vocabulary),
                        ncol = length(tokenized_corpus),
                        dimnames = list(vocabulary, doc_labels))

  for (i in seq_along(tokenized_corpus)) {
    counts <- table(tokenized_corpus[[i]])
    bow_matrix[names(counts), i] <- as.numeric(counts)
  }

  df_bow_matrix <- as.data.frame(bow_matrix)

  return(list(matrix = bow_matrix, df = df_bow_matrix))
}

get_TFIDF_matrix <- function(nonProcessedTexts,
                              dontRemoveStopWords = FALSE) {

  tokenized_corpus <- get_processed_tokens(nonProcessedTexts, dontRemoveStopWords)

  vocabulary <- sort(unique(unlist(tokenized_corpus)))
  doc_labels <- sapply(tokenized_corpus, paste, collapse = " ")

  n_docs <- length(tokenized_corpus)

  # matriz de contagem (mesma lógica do BOW)
  count_matrix <- matrix(0,
                          nrow = length(vocabulary),
                          ncol = n_docs,
                          dimnames = list(vocabulary, doc_labels))

  for (i in seq_along(tokenized_corpus)) {
    counts <- table(tokenized_corpus[[i]])
    count_matrix[names(counts), i] <- as.numeric(counts)
  }

  # document frequency por termo
  doc_freq <- rowSums(count_matrix > 0)

  # idf suavizado (equivalente ao default do sklearn: smooth_idf=True)
  idf_vec <- log((1 + n_docs) / (1 + doc_freq)) + 1

  # tf-idf bruto (contagem x idf)
  tfidf_matrix <- count_matrix * idf_vec

  # normalização L2 por documento (coluna)
  col_norms <- sqrt(colSums(tfidf_matrix^2))
  col_norms[col_norms == 0] <- 1
  tfidf_matrix <- sweep(tfidf_matrix, 2, col_norms, "/")

  df_tfidf_matrix <- as.data.frame(tfidf_matrix)

  return(list(matrix = tfidf_matrix, df = df_tfidf_matrix))
}

### **Parte 2B - Testando Módulo de criação da matriz BOW + a matriz TF-IDF:**

In [54]:
textos_simples <- list("I love you", "Love")
textos_simples

[[1]]
[1] "I love you"

[[2]]
[1] "Love"

#### **Sub-teste: remoção de stopwords desativada:**

In [61]:
tokens <- get_processed_tokens(
    textos_simples, 
    dontRemoveStopWords = TRUE)
print(tokens)

[[1]]
[1] "i"    "love" "you" 

[[2]]
[1] "love"



In [62]:
resultado_bow <- get_BOW_matrix(
    textos_simples, 
    dontRemoveStopWords = TRUE)

# print(resultado_bow$matrix)
# print(resultado_bow$df)
resultado_bow$matrix

,i love you,love
i,1,0
love,1,1
you,1,0


In [63]:
resultado_tfidf <- get_TFIDF_matrix(
    textos_simples, 
    dontRemoveStopWords = TRUE)
# print(resultado_tfidf$matrix)
# print(resultado_tfidf$df)
resultado_tfidf$matrix

,i love you,love
i,0.6316672,0
love,0.4494364,1
you,0.6316672,0


##### **Explicação do código acima (Remoção de stopwords DESATIVADA):**

---

**Documento 1: "I love you"** $\rightarrow$ tokens: $[\text{i},\ \text{love},\ \text{you}]$

**Documento 2: "love"** $\rightarrow$ tokens: $[\text{love}]$

*(sem remoção de stopwords)*

---

**Passo 1 — Vocabulário**

$$V = \{\text{i},\ \text{love},\ \text{you}\}$$

---

**Passo 2 — Matriz BoW (contagens)**

Linhas = termos, colunas = documentos $\{d_1, d_2\}$, onde $d_1 =$ "i love you" e $d_2 =$ "love":

$$
\text{BoW} =
\begin{array}{c|cc}
 & d_1 & d_2 \\
\hline
\text{i} & 1 & 0 \\
\text{love} & 1 & 1 \\
\text{you} & 1 & 0
\end{array}
$$

Em notação matricial pura:

$$
\text{BoW} =
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
1 & 0
\end{bmatrix}
$$

---

**Passo 3 — Document Frequency ($n_t$)**

$$n_{\text{i}} = 1, \qquad n_{\text{love}} = 2, \qquad n_{\text{you}} = 1$$

---

**Passo 4 — IDF suavizado**

$$\text{IDF}(t) = \ln\!\left(\frac{N+1}{n_t+1}\right) + 1, \qquad N = 2$$

$$\text{IDF}(\text{i}) = \ln\!\left(\frac{2+1}{1+1}\right) + 1 = \ln\!\left(\frac{3}{2}\right) + 1 \approx 0.405465 + 1 = 1.405465$$

$$\text{IDF}(\text{love}) = \ln\!\left(\frac{2+1}{2+1}\right) + 1 = \ln(1) + 1 = 0 + 1 = 1$$

$$\text{IDF}(\text{you}) = \ln\!\left(\frac{2+1}{1+1}\right) + 1 = \ln\!\left(\frac{3}{2}\right) + 1 \approx 0.405465 + 1 = 1.405465$$

---

**Passo 5 — Matriz diagonal IDF**

$$
\Sigma_{\text{IDF}} =
\begin{bmatrix}
1.405465 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1.405465
\end{bmatrix}
$$

(ordem das linhas/colunas segue $V = \{\text{i}, \text{love}, \text{you}\}$)

---

**Passo 6 — TF-IDF bruto**

Cada célula de $\text{BoW}$ é multiplicada pelo IDF do seu termo (linha), ou seja $\text{TFIDF}_{raw} = \Sigma_{\text{IDF}} \cdot \text{BoW}$:

$$
\Sigma_{\text{IDF}} \cdot \text{BoW} =
\begin{bmatrix}
1.405465 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1.405465
\end{bmatrix}
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
1 & 0
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
1.405465 \times 1 & 1.405465 \times 0 \\
1 \times 1 & 1 \times 1 \\
1.405465 \times 1 & 1.405465 \times 0
\end{bmatrix}
=
\begin{bmatrix}
1.405465 & 0 \\
1 & 1 \\
1.405465 & 0
\end{bmatrix}
$$

Ou seja:

$$
\text{TFIDF}_{raw} =
\begin{array}{c|cc}
 & d_1 & d_2 \\
\hline
\text{i} & 1.405465 & 0 \\
\text{love} & 1 & 1 \\
\text{you} & 1.405465 & 0
\end{array}
$$

---

**Passo 7 — Normalização L2 por documento (coluna)**

$$\|\mathbf{d_1}\|_2 = \sqrt{1.405465^2 + 1^2 + 1.405465^2} = \sqrt{1.975334 + 1 + 1.975334} = \sqrt{4.950668} \approx 2.224999$$

$$\|\mathbf{d_2}\|_2 = \sqrt{0^2 + 1^2 + 0^2} = \sqrt{1} = 1$$

---

**Passo 8 — Matriz TF-IDF final (normalizada)**

$$
d_1^{norm} =
\left[
\frac{1.405465}{2.224999},\ \ \frac{1}{2.224999},\ \ \frac{1.405465}{2.224999}
\right]
\approx
\left[0.631668,\ \ 0.449436,\ \ 0.631668\right]
$$

$$
d_2^{norm} =
\left[\frac{0}{1},\ \ \frac{1}{1},\ \ \frac{0}{1}\right]
=
\left[0,\ \ 1,\ \ 0\right]
$$

$$
\text{TFIDF}_{final} =
\begin{array}{c|cc}
 & d_1 \text{ ("i love you")} & d_2 \text{ ("love")} \\
\hline
\text{i} & 0.631668 & 0 \\
\text{love} & 0.449436 & 1 \\
\text{you} & 0.631668 & 0
\end{array}
$$

Em notação matricial pura:

$$
\text{TFIDF}_{final} =
\begin{bmatrix}
0.631668 & 0 \\
0.449436 & 1 \\
0.631668 & 0
\end{bmatrix}
$$

Essa matriz final corresponde exatamente à saída do R (`resultado_tfidf$matrix`), com termos nas linhas e documentos nas colunas.

#### **Sub-teste: remoção de stopwords ativada:**

In [64]:
tokens <- get_processed_tokens(
    textos_simples, 
    dontRemoveStopWords = FALSE)
print(tokens)

[[1]]
[1] "love"

[[2]]
[1] "love"



In [72]:
resultado_bow <- get_BOW_matrix(
    textos_simples, 
    dontRemoveStopWords = FALSE)
    
resultado_bow$matrix

,love,love
love,1,1


In [76]:
resultado_tfidf <- get_TFIDF_matrix(
    textos_simples, 
    dontRemoveStopWords = FALSE)

resultado_tfidf$matrix

,love,love
love,1,1
